# Workspace Registry Migration

Migrate MLflow models between workspaces. Supports **WS→WS**, **UC→UC**, and **Indirect Transfer**.

**How to use:** Edit the **Configuration** cell below, then **Run All**.

In [0]:
%restart_python

In [0]:
AUTH_MODE        = "pat"               # "pat" or "service_principal"

# If AUTH_MODE is pat
SOURCE_HOST      = ""                   #Host URL
SOURCE_TOKEN     = ""                   # Host workspace PAT Token

# If AUTH_MODE is service_principal
SOURCE_HOST      = ""                  #Host URL
SP_CLIENT_ID     = ""                  # Service Principal client ID (SP auth only)
SP_CLIENT_SECRET = ""                  # Service Principal client secret (SP auth only)

# --- Migration Direction ---------------------------------------------
# Indicate if Models in source/target belong to legacy workspace registery or unity catalog.
SOURCE_REGISTRY  = "workspace"
TARGET_REGISTRY  = "workspace"

# --- Models to Migrate -----------------------------------------------
# Migrate specific models by name:
# 1. For sepecefic models, use model names like "001_DEMO_user01"
# 2. For bulk model scan, use MODEL_NAMES = []
MODEL_NAMES = [
    #"001_DEMO_user01"
]

# --- Artifact Staging ------------------------------------------------
# Temporary directory for downloading model artifacts before uploading to target.
ARTIFACT_TEMP_DIR = "/tmp/ws_export_bundle" # eg. "gs://.., /dbfs/tmp/.., /Volumes/.., /tmp/.."

# --- Migration Mode --------------------------------------------------
# "direct"  — standard end-to-end migration (default, unchanged behavior)
MIGRATION_MODE = "direct"

# --- Options ---------------------------------------------------------
MODEL_NAME_PREFIX      = None #"ws_test_"        # Prefix for workspace target names (auto if blank)
TRACKING_TABLE         = None  # Delta tracking table
INCLUDE_ARTIFACTS      = True          # Copy model artifacts
INCLUDE_DELETED        = False         # Include soft-deleted runs
CREATE_DUMMY_VERSIONS  = True          # Placeholder versions for deleted source runs
BATCH_SIZE             = 10            # Parallel batch size

In [0]:
AUTH_MODE        = "pat"               # "pat" or "service_principal"

# If AUTH_MODE is pat
SOURCE_HOST      = ""   #Host URL
SOURCE_TOKEN     = ""   # Host workspace PAT Token

# If AUTH_MODE is service_principal
SOURCE_HOST      = ""   #Host URL
SP_CLIENT_ID     = ""                  # Service Principal client ID (SP auth only)
SP_CLIENT_SECRET = ""                  # Service Principal client secret (SP auth only)

# --- Migration Direction ---------------------------------------------
SOURCE_REGISTRY  = "uc"         # "workspace" or "uc"
TARGET_REGISTRY  = "uc"         # "workspace" or "uc"

# --- Models to Migrate -----------------------------------------------
MODEL_NAMES = [
    "catalog.schema.model"
]

# For bulk UC model migration :
INCLUDE_CATALOGS = []         # Only scan these catalogs
EXCLUDE_CATALOGS = []         # Catalogs to skip during bulk scan
EXCLUDE_SCHEMAS  = []         # Schemas to skip ("catalog.schema" format)

# --- Target Settings -------------------------------------------------
UC_TARGET_CATALOG  = ""                # Override target catalog
UC_TARGET_SCHEMA   = ""                # Override target schema
MODEL_NAME_PREFIX  = ""

# --- Artifact Staging ------------------------------------------------
ARTIFACT_TEMP_DIR = "/tmp/ws_export_bundle"

# --- Options ---------------------------------------------------------
MIGRATION_MODE         = "direct"
TRACKING_TABLE         = None
INCLUDE_ARTIFACTS      = True
INCLUDE_DELETED        = False
CREATE_DUMMY_VERSIONS  = True
BATCH_SIZE             = 10

In [0]:
AUTH_MODE        = "pat"               # "pat" or "service_principal"

# If AUTH_MODE is pat
SOURCE_HOST      = ""   #Host URL
SOURCE_TOKEN     = ""   # Host workspace PAT Token

# If AUTH_MODE is service_principal
SOURCE_HOST      = ""   #Host URL
SP_CLIENT_ID     = ""                  # Service Principal client ID (SP auth only)
SP_CLIENT_SECRET = ""                  # Service Principal client secret (SP auth only)

# --- Migration Direction ---------------------------------------------
SOURCE_REGISTRY  = "workspace"         # "workspace" or "uc"
TARGET_REGISTRY  = "workspace"         # "workspace" or "uc"

# --- Models to Migrate -----------------------------------------------
MODEL_NAMES = [
    # "001_DEMO_user01",
]

# For bulk UC model migration :
INCLUDE_CATALOGS = []
EXCLUDE_CATALOGS = []
EXCLUDE_SCHEMAS  = []

# --- Target Settings -------------------------------------------------
UC_TARGET_CATALOG  = ""
UC_TARGET_SCHEMA   = ""
MODEL_NAME_PREFIX  = ""

# --- Artifact Staging ------------------------------------------------
ARTIFACT_TEMP_DIR = "/tmp/ws_export_bundle"

# --- Migration Mode --------------------------------------------------
# "direct"  — standard end-to-end migration (default)
# "export"  — discover + download artifacts + write JSON manifests (no target ops)
# "import"  — read manifests + artifacts from ARTIFACT_TEMP_DIR, create on target
MIGRATION_MODE = "export"

# --- Options ---------------------------------------------------------
TRACKING_TABLE         = None
INCLUDE_ARTIFACTS      = True
INCLUDE_DELETED        = False
CREATE_DUMMY_VERSIONS  = True
BATCH_SIZE             = 10

In [0]:
# Project setup + execute (do not modify)
import importlib.util, os, sys

_root = "/Workspace" + os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)

# Load bootstrap — try FUSE first; if broken, download one file via REST
_bs_file = os.path.join(_root, "workspace_registry_migrator", "bootstrap.py")
try:
    with open(_bs_file, "r") as _f:
        _f.read(1)  # probe FUSE
    _bs_source = _bs_file
except (OSError, IOError):
    from databricks.sdk import WorkspaceClient
    _code = WorkspaceClient().workspace.download(
        _bs_file.removeprefix("/Workspace")
    ).read()
    _bs_source = "/tmp/_ws_bootstrap.py"
    with open(_bs_source, "wb") as _f:
        _f.write(_code)

_spec = importlib.util.spec_from_file_location("_bootstrap", _bs_source)
_bs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_bs)
_bs.ensure_package_importable(_root)

from workspace_registry_migrator.notebook_helpers import execute_migration

# Use globals().get() for scenario-specific settings so any config cell works
_g = globals().get
_migration_df = execute_migration(
    source_host=SOURCE_HOST, source_token=SOURCE_TOKEN,
    auth_mode=AUTH_MODE, sp_client_id=_g("SP_CLIENT_ID", ""), sp_client_secret=_g("SP_CLIENT_SECRET", ""),
    source_registry=SOURCE_REGISTRY, target_registry=TARGET_REGISTRY,
    model_names=MODEL_NAMES,
    include_catalogs=_g("INCLUDE_CATALOGS", []),
    exclude_catalogs=_g("EXCLUDE_CATALOGS", []), exclude_schemas=_g("EXCLUDE_SCHEMAS", []),
    uc_target_catalog=_g("UC_TARGET_CATALOG", ""), uc_target_schema=_g("UC_TARGET_SCHEMA", ""),
    model_name_prefix=_g("MODEL_NAME_PREFIX", ""), tracking_table=TRACKING_TABLE,
    include_artifacts=INCLUDE_ARTIFACTS, include_deleted_runs=INCLUDE_DELETED,
    create_dummy_versions=CREATE_DUMMY_VERSIONS, batch_size=BATCH_SIZE,
    artifact_temp_dir=ARTIFACT_TEMP_DIR,
    migration_mode=_g("MIGRATION_MODE", "direct"),
)
if _migration_df is not None:
    display(_migration_df)
